# Membrane–actin Brownian ratchet — from boundary baselines to a coupled flexible membrane

_Investigation `membrane-actin-ratchet` — coder reproduction notebook._

**Question.** Can a particle-based actin Brownian ratchet (ReaDDy) push against a deformable membrane
(Mem3DG) through a physically faithful bidirectional coupler, and does increasing boundary
realism — fixed wall → rigid movable object → flexible membrane — reproduce the
force/velocity signatures that distinguish a flexible-membrane ratchet (Inoue 2015) from
the classical rigid-wall regime (Peskin 1993)?

An investigation is a sequence of studies that together answer one research question. This
one asks whether a coupled actin (ReaDDy) + membrane (Mem3DG) composite behaves like a real
flexible-membrane Brownian ratchet, by comparing three boundary models and stress-testing the
coupler that powers the flexible one. It merges two earlier framings — a boundary-condition
staircase and a coupling-mechanism deep-dive — into one arc, because the staircase's top rung
*is* the coupler the deep-dive validates.

---

**How to use this notebook**

1. **Run ▸ Run All Cells** to reproduce every study end to end.
2. To explore, edit a study's **Parameters** cell, then re-run from its **Run** section — the figures regenerate with your values.
3. The **Setup** cell at the top is collapsed scaffolding (environment + render helpers); expand it only to inspect. Set `RERUN = False` there to render the committed `runs.db` without re-simulating.

The prose states the **question and parameters**; the figures each run produces are the results.


In [ ]:
"""Setup — environment & helpers (runs automatically; open only to inspect).

Generated by vivarium-workbench (notebook_export). This cell resolves the repo,
registers the workspace Core, and defines the render helpers; the science lives
in the sections below. Each study is re-run live with the workspace's own
process-bigraph protocol and its figures rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-membrane-actin-composite/viva-membrane-actin-composite').is_dir():
    REPO = Path('/home/runner/work/viva-membrane-actin-composite/viva-membrane-actin-composite')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# Keep the output clean: composite discovery and the pydantic models emit
# benign UserWarnings (optional packages it skips, a field-name shadow) and the
# simulators emit DeprecationWarnings — none are actionable from a reproduction
# notebook, so silence those two categories. Remove this to see them.
import warnings as _warnings
_warnings.filterwarnings("ignore", category=UserWarning)
_warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- standard process-bigraph protocol: register the workspace's Core ---
from pbg_membrane_actin_composite.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

def materialize_composite(spec_id, overrides=None):
    """Build a composite into an editable payload {name, description,
    parameters, state, _overrides, _id}. Works for @composite_generator
    functions via the process-bigraph registry (the common case); ``state``
    is the built document (process nodes at top level)."""
    import importlib
    from process_bigraph.composite_generator import (
        _REGISTRY, apply_core_extensions, build_generator)
    try:
        from process_bigraph.composite_generator import discover_generators
        discover_generators()
    except Exception:
        pass
    entry = _REGISTRY.get(spec_id)
    if entry is None:
        # editable installs are invisible to packages_distributions, so
        # discover alone can miss them — prime the module the id names.
        try:
            importlib.import_module(spec_id.rsplit('.', 1)[0])
            from process_bigraph.composite_generator import discover_generators as _dg
            _dg()
        except Exception:
            pass
        entry = _REGISTRY.get(spec_id)
    if entry is None:
        raise KeyError('composite not registered: ' + repr(spec_id))
    params = dict(entry.parameters or {})
    ov = {k: (v.get('default') if isinstance(v, dict) else None)
          for k, v in params.items()}
    if overrides:
        ov.update({k: val for k, val in overrides.items() if k in params})
    _c = apply_core_extensions(entry, core)
    try:
        doc = build_generator(entry, ov or None, core=_c)
    except TypeError:
        doc = entry.func(_c, **(ov or {}))
    return {'name': entry.name, 'description': entry.description,
            'parameters': params, 'state': doc, '_overrides': ov, '_id': spec_id}

import base64 as _b64, pathlib as _pl
_STATIC_VIZ_SCHEMES = ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:')
def _render_viz_process(address, config):
    """Render a Visualization process/step (e.g. ``local:<Class>``) to
    HTML: wrap it as a one-step composite and run it once — the viz writes
    Plotly HTML to its ``html`` output port. Mirrors the workbench's own
    env-worker viz path. Self-running vizes need no run data."""
    from process_bigraph import Composite
    _doc = {
        'inputs_store': {}, 'output_store': '',
        'visualization': {
            '_type': 'step', 'address': address,
            'config': {k: v for k, v in (config or {}).items()
                       if k not in ('inputs_map', 'sources')},
            'inputs': {}, 'outputs': {'html': ['output_store']},
        },
    }
    _comp = Composite({'state': _doc}, core=core)
    _comp.run(1)
    _html = _comp.state.get('output_store')
    if isinstance(_html, dict):
        _html = _html.get('value') or _html.get('_value') or ''
    return _html if isinstance(_html, str) else ''
def _render_one(address, config, runs_db, study_yaml, overrides=None):
    """Generic figure renderer (no workspace render_study_viz.py).
    A static ``image:``/``file:`` (or bare-path) address embeds the file;
    any other address (``local:<VizProcess>``) is run as a Visualization
    and its HTML returned. ``overrides`` (the composite's edited parameter
    dict) is merged into the viz config first so a self-running figure
    reflects edited parameters; a viz that does not accept those keys
    falls back to the base config."""
    addr = str(address or '')
    _head = addr.split('/', 1)[0]
    _is_static = (not addr) or (':' not in _head) or addr.startswith(_STATIC_VIZ_SCHEMES)
    if not _is_static:
        if core is None:
            return f'<p style="color:#b91c1c">no core to render {address}</p>'
        _base = {k: v for k, v in (config or {}).items()}
        _tries = ([{**_base, **overrides}] if overrides else []) + [_base]
        _err = ''
        for _cfg in _tries:
            try:
                _h = _render_viz_process(addr, _cfg)
            except Exception as _e:
                _err = (f'<p style="color:#b91c1c">visualization {address} not renderable yet: '
                        f'{type(_e).__name__}: {_e}</p>')
                continue
            return _h or f'<p style="color:#6b7280">{address} produced no HTML</p>'
        return _err
    for _scheme in _STATIC_VIZ_SCHEMES:
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Rung 1 · Fixed boundary (`fixed-boundary`)

**Question.** Does the composite reproduce the rigid-wall Peskin-1993 regime — Brownian-ratchet kinetics against an immovable boundary — to within numerical tolerance, when barrier_kind is "fixed"?

**Purpose.** The coupler publishes a fixed wall_z to ReaDDy; actin filaments confined under the wall accumulate contact_force; barrier_velocity stays at zero by construction.

**Claim.** OBSERVED (partial pass): against an immovable wall mean barrier_velocity = 0 (within ±1e-3) while contact force accumulates. Quantitative match to the Peskin ratchet rate still pending a digitized reference.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `fixed-boundary` | `pbg_membrane_actin_composite.composites.rung1_fixed_boundary` | 5 | — |
| `high-growth` | `pbg_membrane_actin_composite.composites.rung1_fixed_boundary` | 5 | growth_rate=8.0 |
| `low-growth` | `pbg_membrane_actin_composite.composites.rung1_fixed_boundary` | 5 | growth_rate=1.0 |
| `rung1-fixed` | `pbg_membrane_actin_composite.composites.rung1_fixed_boundary` | 16 | — |
| `rung1-fixed` | `pbg_membrane_actin_composite.composites.rung1_fixed_boundary` | 16 | — |
| `rung1-fixed` | `pbg_membrane_actin_composite.composites.rung1_fixed_boundary` | 16 | — |
| `rung1-fixed` | `pbg_membrane_actin_composite.composites.rung1_fixed_boundary` | 16 | — |
| `rung1-fixed` | `pbg_membrane_actin_composite.composites.rung1_fixed_boundary` | 16 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `pbg_membrane_actin_composite.composites.rung1_fixed_boundary`** — `spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary` (static spec file, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary = load_spec(REPO / 'pbg_membrane_actin_composite/composites/rung1_fixed_boundary.composite.yaml')
describe_spec(spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary)

In [ ]:
# === Edit parameters for composite 'rung1-fixed-boundary' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'actin_sim'  (local:ReaDDyProcess)
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['interval'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['box_size'] = [4.0, 4.0, 4.0]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['periodic'] = [False, False, False]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['species']['G'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['reactions'] = []
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['potentials'][0]['type'] = 'harmonic_repulsion'
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['potentials'][0]['species1'] = 'G'
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['potentials'][0]['species2'] = 'G'
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['potentials'][0]['force_constant'] = 10.0
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['potentials'][0]['interaction_distance'] = 0.4
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_particles']['G'][0] = [-0.6000000000000001, -0.6000000000000001, -1.8]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_particles']['G'][1] = [-0.2, -0.6000000000000001, -1.8]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_particles']['G'][2] = [0.2, -0.6000000000000001, -1.8]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_particles']['G'][3] = [0.6000000000000001, -0.6000000000000001, -1.8]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_particles']['G'][4] = [-0.6000000000000001, -0.2, -1.8]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_particles']['G'][5] = [-0.2, -0.2, -1.8]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_particles']['G'][6] = [0.2, -0.2, -1.8]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_particles']['G'][7] = [0.6000000000000001, -0.2, -1.8]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['topology_species']['F'] = 0.05
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['topology_types'] = ['filament']
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['topology_bonds'][0]['type1'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['topology_bonds'][0]['type2'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['topology_bonds'][0]['force_constant'] = 200.0
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['topology_bonds'][0]['length'] = 0.4
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['topology_angles'][0]['type1'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['topology_angles'][0]['type2'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['topology_angles'][0]['type3'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['topology_angles'][0]['force_constant'] = 30.0
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['topology_angles'][0]['equilibrium_angle'] = 3.14159
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][0]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][0]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][0]['positions'][0] = [-0.4, -0.4, -1.5000000000000002]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][0]['positions'][1] = [-0.4, -0.4, -1.1]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][0]['positions'][2] = [-0.4, -0.4, -0.7]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][0]['positions'][3] = [-0.4, -0.4, -0.3]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][1]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][1]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][1]['positions'][0] = [0.4, -0.4, -1.5000000000000002]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][1]['positions'][1] = [0.4, -0.4, -1.1]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][1]['positions'][2] = [0.4, -0.4, -0.7]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][1]['positions'][3] = [0.4, -0.4, -0.3]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][2]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][2]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][2]['positions'][0] = [-0.4, 0.4, -1.5000000000000002]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][2]['positions'][1] = [-0.4, 0.4, -1.1]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][2]['positions'][2] = [-0.4, 0.4, -0.7]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][2]['positions'][3] = [-0.4, 0.4, -0.3]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][3]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][3]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][3]['positions'][0] = [0.4, 0.4, -1.5000000000000002]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][3]['positions'][1] = [0.4, 0.4, -1.1]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][3]['positions'][2] = [0.4, 0.4, -0.7]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][3]['positions'][3] = [0.4, 0.4, -0.3]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][4]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][4]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][4]['positions'][0] = [-0.4, 1.2000000000000002, -1.5000000000000002]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][4]['positions'][1] = [-0.4, 1.2000000000000002, -1.1]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][4]['positions'][2] = [-0.4, 1.2000000000000002, -0.7]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][4]['positions'][3] = [-0.4, 1.2000000000000002, -0.3]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][5]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][5]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][5]['positions'][0] = [0.4, 1.2000000000000002, -1.5000000000000002]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][5]['positions'][1] = [0.4, 1.2000000000000002, -1.1]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][5]['positions'][2] = [0.4, 1.2000000000000002, -0.7]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['initial_topologies'][5]['positions'][3] = [0.4, 1.2000000000000002, -0.3]
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['timestep'] = 0.005
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['actin_sim']['config']['observe_stride'] = 10

# process 'coupler'  (local:BrownianRatchetCoupler)
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['coupler']['interval'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['coupler']['config']['closed_loop'] = True
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['coupler']['config']['coupling_mode'] = 'planar'
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['coupler']['config']['barrier_kind'] = 'fixed'
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['coupler']['config']['barrier_initial_z'] = 0.0
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['coupler']['config']['barrier_drag'] = 5.0
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['coupler']['config']['contact_threshold'] = 0.4
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['coupler']['config']['force_constant'] = 2.0
spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary['state']['coupler']['config']['osmotic_force_scale'] = 0.02

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fixed-boundary ===
STUDY = 'fixed-boundary'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_fixed_boundary = 5
INTERVAL_fixed_boundary = 1.0
STEPS_high_growth = 5
INTERVAL_high_growth = 1.0
STEPS_low_growth = 5
INTERVAL_low_growth = 1.0
STEPS_rung1_fixed = 16
INTERVAL_rung1_fixed = 1.0
STEPS_rung1_fixed = 16
INTERVAL_rung1_fixed = 1.0
STEPS_rung1_fixed = 16
INTERVAL_rung1_fixed = 1.0
STEPS_rung1_fixed = 16
INTERVAL_rung1_fixed = 1.0
STEPS_rung1_fixed = 16
INTERVAL_rung1_fixed = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary, {'interval': INTERVAL_fixed_boundary}, core=core)
            _comp.run(STEPS_fixed_boundary)
        print('ran', 'fixed-boundary', f'(pbg_membrane_actin_composite.composites.rung1_fixed_boundary) for', STEPS_fixed_boundary, 'steps')
    except Exception as _e:
        print(f"fixed-boundary not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary, {'interval': INTERVAL_high_growth}, core=core)
            _comp.run(STEPS_high_growth)
        print('ran', 'high-growth', f'(pbg_membrane_actin_composite.composites.rung1_fixed_boundary) for', STEPS_high_growth, 'steps')
    except Exception as _e:
        print(f"high-growth not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary, {'interval': INTERVAL_low_growth}, core=core)
            _comp.run(STEPS_low_growth)
        print('ran', 'low-growth', f'(pbg_membrane_actin_composite.composites.rung1_fixed_boundary) for', STEPS_low_growth, 'steps')
    except Exception as _e:
        print(f"low-growth not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary, {'interval': INTERVAL_rung1_fixed}, core=core)
            _comp.run(STEPS_rung1_fixed)
        print('ran', 'rung1-fixed', f'(pbg_membrane_actin_composite.composites.rung1_fixed_boundary) for', STEPS_rung1_fixed, 'steps')
    except Exception as _e:
        print(f"rung1-fixed not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary, {'interval': INTERVAL_rung1_fixed}, core=core)
            _comp.run(STEPS_rung1_fixed)
        print('ran', 'rung1-fixed', f'(pbg_membrane_actin_composite.composites.rung1_fixed_boundary) for', STEPS_rung1_fixed, 'steps')
    except Exception as _e:
        print(f"rung1-fixed not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary, {'interval': INTERVAL_rung1_fixed}, core=core)
            _comp.run(STEPS_rung1_fixed)
        print('ran', 'rung1-fixed', f'(pbg_membrane_actin_composite.composites.rung1_fixed_boundary) for', STEPS_rung1_fixed, 'steps')
    except Exception as _e:
        print(f"rung1-fixed not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary, {'interval': INTERVAL_rung1_fixed}, core=core)
            _comp.run(STEPS_rung1_fixed)
        print('ran', 'rung1-fixed', f'(pbg_membrane_actin_composite.composites.rung1_fixed_boundary) for', STEPS_rung1_fixed, 'steps')
    except Exception as _e:
        print(f"rung1-fixed not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung1_fixed_boundary, {'interval': INTERVAL_rung1_fixed}, core=core)
            _comp.run(STEPS_rung1_fixed)
        print('ran', 'rung1-fixed', f'(pbg_membrane_actin_composite.composites.rung1_fixed_boundary) for', STEPS_rung1_fixed, 'steps')
    except Exception as _e:
        print(f"rung1-fixed not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**gap-contact-decision**


In [ ]:
# gap-contact-decision
show_viz(_render_one('local:GapContactDecision', {'study_slug': 'fixed-boundary'}, RUNS_DB, STUDY_YAML))

**coupling-trace**


In [ ]:
# coupling-trace
show_viz(_render_one('local:CouplingTrace', {}, RUNS_DB, STUDY_YAML))

**population-trace**


In [ ]:
# population-trace
show_viz(_render_one('local:PopulationTrace', {}, RUNS_DB, STUDY_YAML))

**barrier-kinematics**


In [ ]:
# barrier-kinematics
show_viz(_render_one('local:BarrierKinematics', {}, RUNS_DB, STUDY_YAML))

**ratchet-event-rate**


In [ ]:
# ratchet-event-rate
show_viz(_render_one('local:RatchetEventRate', {}, RUNS_DB, STUDY_YAML))

**particles-3d**


In [ ]:
# particles-3d
show_viz(_render_one('local:Particles3D', {'study_slug': 'fixed-boundary'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| fv-matches-mogilner-baseline | kind=listener_path observable=barrier_velocity reduce=mean | op in_range lo -0.001 hi 0.001 |


## Study: Rung 2 · Rigid movable boundary (`rigid-movable-boundary`)

**Question.** With barrier_kind="rigid_movable", does the wall translate at a velocity proportional to the mean contact force (drag-balance Peskin 1993 setup), and does the stall force remain bounded?

**Purpose.** The wall obeys dz/dt = F/drag while staying planar (no deformation). Same actin model as the fixed-boundary rung; only the barrier's response changes.

**Claim.** OBSERVED: a drag-loaded movable wall settles to a non-zero velocity. Whether the stall force stays within the Peskin-1993 bound is NOT yet checked.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `high-drag` | `pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary` | 5 | barrier_drag=16.0 |
| `low-drag` | `pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary` | 5 | barrier_drag=2.0 |
| `rigid-movable-boundary` | `pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary` | 5 | — |
| `rung2-rigid-movable` | `pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary` | 16 | — |
| `rung2-rigid-movable` | `pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary` | 16 | — |
| `rung2-rigid-movable` | `pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary` | 16 | — |
| `rung2-rigid-movable` | `pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary` | 16 | — |
| `rung2-rigid-movable` | `pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary` | 16 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary`** — `spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary` (static spec file, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary = load_spec(REPO / 'pbg_membrane_actin_composite/composites/rung2_rigid_movable_boundary.composite.yaml')
describe_spec(spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary)

In [ ]:
# === Edit parameters for composite 'rung2-rigid-movable-boundary' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'actin_sim'  (local:ReaDDyProcess)
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['interval'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['box_size'] = [4.0, 4.0, 4.0]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['periodic'] = [False, False, False]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['species']['G'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['reactions'] = []
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['potentials'][0]['type'] = 'harmonic_repulsion'
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['potentials'][0]['species1'] = 'G'
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['potentials'][0]['species2'] = 'G'
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['potentials'][0]['force_constant'] = 10.0
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['potentials'][0]['interaction_distance'] = 0.4
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_particles']['G'][0] = [-0.6000000000000001, -0.6000000000000001, -1.8]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_particles']['G'][1] = [-0.2, -0.6000000000000001, -1.8]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_particles']['G'][2] = [0.2, -0.6000000000000001, -1.8]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_particles']['G'][3] = [0.6000000000000001, -0.6000000000000001, -1.8]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_particles']['G'][4] = [-0.6000000000000001, -0.2, -1.8]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_particles']['G'][5] = [-0.2, -0.2, -1.8]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_particles']['G'][6] = [0.2, -0.2, -1.8]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_particles']['G'][7] = [0.6000000000000001, -0.2, -1.8]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['topology_species']['F'] = 0.05
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['topology_types'] = ['filament']
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['topology_bonds'][0]['type1'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['topology_bonds'][0]['type2'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['topology_bonds'][0]['force_constant'] = 200.0
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['topology_bonds'][0]['length'] = 0.4
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['topology_angles'][0]['type1'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['topology_angles'][0]['type2'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['topology_angles'][0]['type3'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['topology_angles'][0]['force_constant'] = 30.0
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['topology_angles'][0]['equilibrium_angle'] = 3.14159
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][0]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][0]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][0]['positions'][0] = [-0.4, -0.4, -1.5000000000000002]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][0]['positions'][1] = [-0.4, -0.4, -1.1]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][0]['positions'][2] = [-0.4, -0.4, -0.7]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][0]['positions'][3] = [-0.4, -0.4, -0.3]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][1]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][1]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][1]['positions'][0] = [0.4, -0.4, -1.5000000000000002]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][1]['positions'][1] = [0.4, -0.4, -1.1]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][1]['positions'][2] = [0.4, -0.4, -0.7]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][1]['positions'][3] = [0.4, -0.4, -0.3]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][2]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][2]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][2]['positions'][0] = [-0.4, 0.4, -1.5000000000000002]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][2]['positions'][1] = [-0.4, 0.4, -1.1]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][2]['positions'][2] = [-0.4, 0.4, -0.7]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][2]['positions'][3] = [-0.4, 0.4, -0.3]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][3]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][3]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][3]['positions'][0] = [0.4, 0.4, -1.5000000000000002]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][3]['positions'][1] = [0.4, 0.4, -1.1]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][3]['positions'][2] = [0.4, 0.4, -0.7]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][3]['positions'][3] = [0.4, 0.4, -0.3]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][4]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][4]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][4]['positions'][0] = [-0.4, 1.2000000000000002, -1.5000000000000002]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][4]['positions'][1] = [-0.4, 1.2000000000000002, -1.1]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][4]['positions'][2] = [-0.4, 1.2000000000000002, -0.7]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][4]['positions'][3] = [-0.4, 1.2000000000000002, -0.3]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][5]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][5]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][5]['positions'][0] = [0.4, 1.2000000000000002, -1.5000000000000002]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][5]['positions'][1] = [0.4, 1.2000000000000002, -1.1]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][5]['positions'][2] = [0.4, 1.2000000000000002, -0.7]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['initial_topologies'][5]['positions'][3] = [0.4, 1.2000000000000002, -0.3]
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['timestep'] = 0.005
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['actin_sim']['config']['observe_stride'] = 10

# process 'coupler'  (local:BrownianRatchetCoupler)
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['coupler']['interval'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['coupler']['config']['closed_loop'] = True
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['coupler']['config']['coupling_mode'] = 'planar'
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['coupler']['config']['barrier_kind'] = 'rigid_movable'
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['coupler']['config']['barrier_initial_z'] = 0.0
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['coupler']['config']['barrier_drag'] = 8.0
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['coupler']['config']['contact_threshold'] = 0.4
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['coupler']['config']['force_constant'] = 2.0
spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary['state']['coupler']['config']['osmotic_force_scale'] = 0.02

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: rigid-movable-boundary ===
STUDY = 'rigid-movable-boundary'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_high_drag = 5
INTERVAL_high_drag = 1.0
STEPS_low_drag = 5
INTERVAL_low_drag = 1.0
STEPS_rigid_movable_boundary = 5
INTERVAL_rigid_movable_boundary = 1.0
STEPS_rung2_rigid_movable = 16
INTERVAL_rung2_rigid_movable = 1.0
STEPS_rung2_rigid_movable = 16
INTERVAL_rung2_rigid_movable = 1.0
STEPS_rung2_rigid_movable = 16
INTERVAL_rung2_rigid_movable = 1.0
STEPS_rung2_rigid_movable = 16
INTERVAL_rung2_rigid_movable = 1.0
STEPS_rung2_rigid_movable = 16
INTERVAL_rung2_rigid_movable = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary, {'interval': INTERVAL_high_drag}, core=core)
            _comp.run(STEPS_high_drag)
        print('ran', 'high-drag', f'(pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary) for', STEPS_high_drag, 'steps')
    except Exception as _e:
        print(f"high-drag not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary, {'interval': INTERVAL_low_drag}, core=core)
            _comp.run(STEPS_low_drag)
        print('ran', 'low-drag', f'(pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary) for', STEPS_low_drag, 'steps')
    except Exception as _e:
        print(f"low-drag not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary, {'interval': INTERVAL_rigid_movable_boundary}, core=core)
            _comp.run(STEPS_rigid_movable_boundary)
        print('ran', 'rigid-movable-boundary', f'(pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary) for', STEPS_rigid_movable_boundary, 'steps')
    except Exception as _e:
        print(f"rigid-movable-boundary not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary, {'interval': INTERVAL_rung2_rigid_movable}, core=core)
            _comp.run(STEPS_rung2_rigid_movable)
        print('ran', 'rung2-rigid-movable', f'(pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary) for', STEPS_rung2_rigid_movable, 'steps')
    except Exception as _e:
        print(f"rung2-rigid-movable not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary, {'interval': INTERVAL_rung2_rigid_movable}, core=core)
            _comp.run(STEPS_rung2_rigid_movable)
        print('ran', 'rung2-rigid-movable', f'(pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary) for', STEPS_rung2_rigid_movable, 'steps')
    except Exception as _e:
        print(f"rung2-rigid-movable not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary, {'interval': INTERVAL_rung2_rigid_movable}, core=core)
            _comp.run(STEPS_rung2_rigid_movable)
        print('ran', 'rung2-rigid-movable', f'(pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary) for', STEPS_rung2_rigid_movable, 'steps')
    except Exception as _e:
        print(f"rung2-rigid-movable not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary, {'interval': INTERVAL_rung2_rigid_movable}, core=core)
            _comp.run(STEPS_rung2_rigid_movable)
        print('ran', 'rung2-rigid-movable', f'(pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary) for', STEPS_rung2_rigid_movable, 'steps')
    except Exception as _e:
        print(f"rung2-rigid-movable not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung2_rigid_movable_boundary, {'interval': INTERVAL_rung2_rigid_movable}, core=core)
            _comp.run(STEPS_rung2_rigid_movable)
        print('ran', 'rung2-rigid-movable', f'(pbg_membrane_actin_composite.composites.rung2_rigid_movable_boundary) for', STEPS_rung2_rigid_movable, 'steps')
    except Exception as _e:
        print(f"rung2-rigid-movable not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**fv-phase-portrait**


In [ ]:
# fv-phase-portrait
show_viz(_render_one('local:FVPhasePortrait', {'study_slug': 'rigid-movable-boundary'}, RUNS_DB, STUDY_YAML))

**coupling-trace**


In [ ]:
# coupling-trace
show_viz(_render_one('local:CouplingTrace', {}, RUNS_DB, STUDY_YAML))

**population-trace**


In [ ]:
# population-trace
show_viz(_render_one('local:PopulationTrace', {}, RUNS_DB, STUDY_YAML))

**barrier-kinematics**


In [ ]:
# barrier-kinematics
show_viz(_render_one('local:BarrierKinematics', {}, RUNS_DB, STUDY_YAML))

**force-velocity-scatter**


In [ ]:
# force-velocity-scatter
show_viz(_render_one('local:ForceVelocityScatter', {}, RUNS_DB, STUDY_YAML))

**ratchet-event-rate**


In [ ]:
# ratchet-event-rate
show_viz(_render_one('local:RatchetEventRate', {}, RUNS_DB, STUDY_YAML))

**particles-3d**


In [ ]:
# particles-3d
show_viz(_render_one('local:Particles3D', {'study_slug': 'rigid-movable-boundary'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| stall-force-within-peskin-bound | kind=listener_path observable=barrier_velocity reduce=mean | op in_range lo 0.001 hi 5.0 |


## Study: Coupler A · Actin→membrane force handshake (`actin-to-membrane-force-handshake`)

**Question.** In the closed-loop flexible composite, is the force the actin side reports at the interface equal-and-opposite to the force the membrane integrates on its receiving vertices (Newton's 3rd law on the coupling interface)?

**Purpose.** BrownianRatchetCoupler reads ReaDDy tip positions, computes per-contact forces using contact_threshold + force_constant, and applies them as osmotic_offset on Mem3DG. The same per-step force scalar must show up in both the actin-side contact_force observable and the membrane-side integrated impulse, up to numerical tolerance.

**Claim.** OBSERVED (option B, magnitude-conserving): the coupler now sizes the osmotic offset so its integrated membrane force equals the actin contact force; force_residual ≈ 0 every step. CAVEAT: pressure-mediated effective-force handshake, not an independent per-vertex check (that is option A, blocked by a pymem3dg applied-force segfault — tracked).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `rung3_flexible_mem3dg_boundary` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary`** — `spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary` (static spec file, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary = load_spec(REPO / 'pbg_membrane_actin_composite/composites/rung3_flexible_mem3dg_boundary.composite.yaml')
describe_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary)

In [ ]:
# === Edit parameters for composite 'rung3-flexible-mem3dg-boundary' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'actin_sim'  (local:ReaDDyProcess)
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['interval'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['box_size'] = [8.0, 8.0, 8.0]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['periodic'] = [False, False, False]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['species']['G'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['reactions'] = []
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['potentials'][0]['type'] = 'harmonic_repulsion'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['potentials'][0]['species1'] = 'G'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['potentials'][0]['species2'] = 'G'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['potentials'][0]['force_constant'] = 10.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['potentials'][0]['interaction_distance'] = 0.4
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][0] = [-0.44999999999999996, -0.44999999999999996, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][1] = [-0.15, -0.44999999999999996, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][2] = [0.15, -0.44999999999999996, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][3] = [0.44999999999999996, -0.44999999999999996, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][4] = [-0.44999999999999996, -0.15, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][5] = [-0.15, -0.15, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][6] = [0.15, -0.15, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][7] = [0.44999999999999996, -0.15, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_species']['F'] = 0.05
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_types'] = ['filament']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_bonds'][0]['type1'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_bonds'][0]['type2'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_bonds'][0]['force_constant'] = 200.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_bonds'][0]['length'] = 0.4
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_angles'][0]['type1'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_angles'][0]['type2'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_angles'][0]['type3'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_angles'][0]['force_constant'] = 30.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_angles'][0]['equilibrium_angle'] = 3.14159
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][0]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][0]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][0]['positions'][0] = [0.11055415967851333, 0.16666666666666669, 0.0]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][0]['positions'][1] = [0.33166247903554, 0.5000000000000001, 0.0]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][0]['positions'][2] = [0.5527707983925666, 0.8333333333333334, 0.0]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][0]['positions'][3] = [0.7738791177495933, 1.1666666666666667, 0.0]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][1]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][1]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][1]['positions'][0] = [-0.12771603607517107, 0.1, 0.11699835096806109]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][1]['positions'][1] = [-0.3831481082255132, 0.30000000000000004, 0.3509950529041833]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][1]['positions'][2] = [-0.6385801803758553, 0.5, 0.5849917548403054]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][1]['positions'][3] = [-0.8940122525261975, 0.7000000000000001, 0.8189884567764276]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][2]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][2]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][2]['positions'][0] = [0.01724058541736129, 0.03333333333333335, -0.19644757851232364]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][2]['positions'][1] = [0.051721756252083875, 0.10000000000000006, -0.589342735536971]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][2]['positions'][2] = [0.08620292708680645, 0.16666666666666674, -0.9822378925616182]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][2]['positions'][3] = [0.12068409792152904, 0.23333333333333345, -1.3751330495862657]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][3]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][3]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][3]['positions'][0] = [0.11998576148964023, -0.033333333333333305, 0.15650017868564897]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][3]['positions'][1] = [0.3599572844689207, -0.09999999999999992, 0.46950053605694697]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][3]['positions'][2] = [0.5999288074482011, -0.16666666666666652, 0.7825008934282448]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][3]['positions'][3] = [0.8399003304274816, -0.23333333333333314, 1.0955012507995427]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][4]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][4]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][4]['positions'][0] = [-0.1705573787464552, -0.1, -0.030169198781840885]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][4]['positions'][1] = [-0.5116721362393657, -0.30000000000000004, -0.09050759634552266]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][4]['positions'][2] = [-0.852786893732276, -0.5, -0.15084599390920442]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][4]['positions'][3] = [-1.1939016512251865, -0.7000000000000001, -0.2111843914728862]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][5]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][5]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][5]['positions'][0] = [0.09328065759228087, -0.16666666666666666, -0.059337518833987996]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][5]['positions'][1] = [0.27984197277684264, -0.5, -0.178012556501964]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][5]['positions'][2] = [0.4664032879614043, -0.8333333333333333, -0.29668759416993995]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][5]['positions'][3] = [0.652964603145966, -1.1666666666666667, -0.415362631837916]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['timestep'] = 0.005
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['observe_stride'] = 10

# process 'coupler'  (local:BrownianRatchetCoupler)
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['interval'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['closed_loop'] = True
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['coupling_mode'] = 'spherical'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['barrier_kind'] = 'flexible'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['barrier_initial_z'] = 2.5
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['barrier_drag'] = 5.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['contact_threshold'] = 0.3
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['force_constant'] = 5.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['osmotic_force_scale'] = 0.05

# process 'membrane_sim'  (local:Mem3DGProcess)
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['interval'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['mesh_type'] = 'icosphere'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['radius'] = 2.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['subdivision'] = 2
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['characteristic_timestep'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['tolerance'] = 1e-09
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['osmotic_model'] = 'constant'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['osmotic_pressure'] = 0.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['tension_modulus'] = 1.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['preferred_area_scale'] = 1.0

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: actin-to-membrane-force-handshake ===
STUDY = 'actin-to-membrane-force-handshake'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_rung3_flexible_mem3dg_boundary = 100
INTERVAL_rung3_flexible_mem3dg_boundary = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_rung3_flexible_mem3dg_boundary}, core=core)
            _comp.run(STEPS_rung3_flexible_mem3dg_boundary)
        print('ran', 'rung3_flexible_mem3dg_boundary', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_rung3_flexible_mem3dg_boundary, 'steps')
    except Exception as _e:
        print(f"rung3_flexible_mem3dg_boundary not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**newton-residual**


In [ ]:
# newton-residual
show_viz(_render_one('local:NewtonResidual', {'study_slug': 'actin-to-membrane-force-handshake'}, RUNS_DB, STUDY_YAML))

**coupling-trace**


In [ ]:
# coupling-trace
show_viz(_render_one('local:CouplingTrace', {}, RUNS_DB, STUDY_YAML))

**backpressure-trace**


In [ ]:
# backpressure-trace
show_viz(_render_one('local:BackpressureTrace', {}, RUNS_DB, STUDY_YAML))

**barrier-kinematics**


In [ ]:
# barrier-kinematics
show_viz(_render_one('local:BarrierKinematics', {}, RUNS_DB, STUDY_YAML))

**energy-budget**


In [ ]:
# energy-budget
show_viz(_render_one('local:EnergyBudget', {}, RUNS_DB, STUDY_YAML))

**ratchet-event-rate**


In [ ]:
# ratchet-event-rate
show_viz(_render_one('local:RatchetEventRate', {}, RUNS_DB, STUDY_YAML))

**schematic-vesicle-3d**


In [ ]:
# schematic-vesicle-3d
show_viz(_render_one('local:SchematicVesicle3D', {}, RUNS_DB, STUDY_YAML))

**particles-3d**


In [ ]:
# particles-3d
show_viz(_render_one('local:Particles3D', {'study_slug': 'actin-to-membrane-force-handshake'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| newton-third-law-holds-on-interface | kind=listener_path observable=contact_force reduce=series | op rolling_cv_below threshold 0.05 |


## Study: Coupler B · Membrane→actin displacement feedback (`membrane-to-actin-displacement-feedback`)

**Question.** Does per-step mesh-position feedback from Mem3DG to ReaDDy preserve mesh quality (no vertex inversion or degenerate triangles) and keep tip particles within their box-potential bounds?

**Purpose.** After every Mem3DG step the coupler publishes updated wall_radius (planar) or membrane_vertex_positions (spherical) to ReaDDy as the new boundary. Aggressive force_constant or osmotic_force_scale can push the boundary past actin tips in one step, violating bond/box potentials.

**Claim.** TARGET (not yet tested): per-step mesh-position feedback preserves mesh quality and keeps tips inside their box potentials. Mesh-quality is currently inferred, not measured.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `aggressive-feedback` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 5 | osmotic_force_scale=0.12 |
| `feedback-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `feedback-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `feedback-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `feedback-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `feedback-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `gentle-feedback` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 5 | osmotic_force_scale=0.02 |
| `membrane-to-actin-displacement-feedback` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 5 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary`** — `spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary` (static spec file, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary = load_spec(REPO / 'pbg_membrane_actin_composite/composites/rung3_flexible_mem3dg_boundary.composite.yaml')
describe_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary)

In [ ]:
# === Edit parameters for composite 'rung3-flexible-mem3dg-boundary' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'actin_sim'  (local:ReaDDyProcess)
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['interval'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['box_size'] = [8.0, 8.0, 8.0]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['periodic'] = [False, False, False]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['species']['G'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['reactions'] = []
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['potentials'][0]['type'] = 'harmonic_repulsion'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['potentials'][0]['species1'] = 'G'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['potentials'][0]['species2'] = 'G'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['potentials'][0]['force_constant'] = 10.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['potentials'][0]['interaction_distance'] = 0.4
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][0] = [-0.44999999999999996, -0.44999999999999996, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][1] = [-0.15, -0.44999999999999996, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][2] = [0.15, -0.44999999999999996, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][3] = [0.44999999999999996, -0.44999999999999996, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][4] = [-0.44999999999999996, -0.15, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][5] = [-0.15, -0.15, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][6] = [0.15, -0.15, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][7] = [0.44999999999999996, -0.15, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_species']['F'] = 0.05
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_types'] = ['filament']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_bonds'][0]['type1'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_bonds'][0]['type2'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_bonds'][0]['force_constant'] = 200.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_bonds'][0]['length'] = 0.4
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_angles'][0]['type1'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_angles'][0]['type2'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_angles'][0]['type3'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_angles'][0]['force_constant'] = 30.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_angles'][0]['equilibrium_angle'] = 3.14159
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][0]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][0]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][0]['positions'][0] = [0.11055415967851333, 0.16666666666666669, 0.0]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][0]['positions'][1] = [0.33166247903554, 0.5000000000000001, 0.0]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][0]['positions'][2] = [0.5527707983925666, 0.8333333333333334, 0.0]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][0]['positions'][3] = [0.7738791177495933, 1.1666666666666667, 0.0]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][1]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][1]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][1]['positions'][0] = [-0.12771603607517107, 0.1, 0.11699835096806109]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][1]['positions'][1] = [-0.3831481082255132, 0.30000000000000004, 0.3509950529041833]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][1]['positions'][2] = [-0.6385801803758553, 0.5, 0.5849917548403054]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][1]['positions'][3] = [-0.8940122525261975, 0.7000000000000001, 0.8189884567764276]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][2]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][2]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][2]['positions'][0] = [0.01724058541736129, 0.03333333333333335, -0.19644757851232364]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][2]['positions'][1] = [0.051721756252083875, 0.10000000000000006, -0.589342735536971]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][2]['positions'][2] = [0.08620292708680645, 0.16666666666666674, -0.9822378925616182]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][2]['positions'][3] = [0.12068409792152904, 0.23333333333333345, -1.3751330495862657]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][3]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][3]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][3]['positions'][0] = [0.11998576148964023, -0.033333333333333305, 0.15650017868564897]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][3]['positions'][1] = [0.3599572844689207, -0.09999999999999992, 0.46950053605694697]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][3]['positions'][2] = [0.5999288074482011, -0.16666666666666652, 0.7825008934282448]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][3]['positions'][3] = [0.8399003304274816, -0.23333333333333314, 1.0955012507995427]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][4]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][4]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][4]['positions'][0] = [-0.1705573787464552, -0.1, -0.030169198781840885]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][4]['positions'][1] = [-0.5116721362393657, -0.30000000000000004, -0.09050759634552266]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][4]['positions'][2] = [-0.852786893732276, -0.5, -0.15084599390920442]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][4]['positions'][3] = [-1.1939016512251865, -0.7000000000000001, -0.2111843914728862]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][5]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][5]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][5]['positions'][0] = [0.09328065759228087, -0.16666666666666666, -0.059337518833987996]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][5]['positions'][1] = [0.27984197277684264, -0.5, -0.178012556501964]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][5]['positions'][2] = [0.4664032879614043, -0.8333333333333333, -0.29668759416993995]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][5]['positions'][3] = [0.652964603145966, -1.1666666666666667, -0.415362631837916]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['timestep'] = 0.005
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['observe_stride'] = 10

# process 'coupler'  (local:BrownianRatchetCoupler)
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['interval'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['closed_loop'] = True
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['coupling_mode'] = 'spherical'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['barrier_kind'] = 'flexible'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['barrier_initial_z'] = 2.5
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['barrier_drag'] = 5.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['contact_threshold'] = 0.3
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['force_constant'] = 5.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['osmotic_force_scale'] = 0.05

# process 'membrane_sim'  (local:Mem3DGProcess)
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['interval'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['mesh_type'] = 'icosphere'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['radius'] = 2.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['subdivision'] = 2
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['characteristic_timestep'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['tolerance'] = 1e-09
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['osmotic_model'] = 'constant'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['osmotic_pressure'] = 0.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['tension_modulus'] = 1.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['preferred_area_scale'] = 1.0

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: membrane-to-actin-displacement-feedback ===
STUDY = 'membrane-to-actin-displacement-feedback'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_aggressive_feedback = 5
INTERVAL_aggressive_feedback = 1.0
STEPS_feedback_flexible = 16
INTERVAL_feedback_flexible = 1.0
STEPS_feedback_flexible = 16
INTERVAL_feedback_flexible = 1.0
STEPS_feedback_flexible = 16
INTERVAL_feedback_flexible = 1.0
STEPS_feedback_flexible = 16
INTERVAL_feedback_flexible = 1.0
STEPS_feedback_flexible = 16
INTERVAL_feedback_flexible = 1.0
STEPS_gentle_feedback = 5
INTERVAL_gentle_feedback = 1.0
STEPS_membrane_to_actin_displacement_feedback = 5
INTERVAL_membrane_to_actin_displacement_feedback = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_aggressive_feedback}, core=core)
            _comp.run(STEPS_aggressive_feedback)
        print('ran', 'aggressive-feedback', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_aggressive_feedback, 'steps')
    except Exception as _e:
        print(f"aggressive-feedback not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_feedback_flexible}, core=core)
            _comp.run(STEPS_feedback_flexible)
        print('ran', 'feedback-flexible', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_feedback_flexible, 'steps')
    except Exception as _e:
        print(f"feedback-flexible not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_feedback_flexible}, core=core)
            _comp.run(STEPS_feedback_flexible)
        print('ran', 'feedback-flexible', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_feedback_flexible, 'steps')
    except Exception as _e:
        print(f"feedback-flexible not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_feedback_flexible}, core=core)
            _comp.run(STEPS_feedback_flexible)
        print('ran', 'feedback-flexible', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_feedback_flexible, 'steps')
    except Exception as _e:
        print(f"feedback-flexible not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_feedback_flexible}, core=core)
            _comp.run(STEPS_feedback_flexible)
        print('ran', 'feedback-flexible', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_feedback_flexible, 'steps')
    except Exception as _e:
        print(f"feedback-flexible not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_feedback_flexible}, core=core)
            _comp.run(STEPS_feedback_flexible)
        print('ran', 'feedback-flexible', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_feedback_flexible, 'steps')
    except Exception as _e:
        print(f"feedback-flexible not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_gentle_feedback}, core=core)
            _comp.run(STEPS_gentle_feedback)
        print('ran', 'gentle-feedback', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_gentle_feedback, 'steps')
    except Exception as _e:
        print(f"gentle-feedback not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_membrane_to_actin_displacement_feedback}, core=core)
            _comp.run(STEPS_membrane_to_actin_displacement_feedback)
        print('ran', 'membrane-to-actin-displacement-feedback', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_membrane_to_actin_displacement_feedback, 'steps')
    except Exception as _e:
        print(f"membrane-to-actin-displacement-feedback not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**coupling-trace**


In [ ]:
# coupling-trace
show_viz(_render_one('local:CouplingTrace', {}, RUNS_DB, STUDY_YAML))

**backpressure-trace**


In [ ]:
# backpressure-trace
show_viz(_render_one('local:BackpressureTrace', {}, RUNS_DB, STUDY_YAML))

**membrane-volume-strain**


In [ ]:
# membrane-volume-strain
show_viz(_render_one('local:MembraneVolumeStrain', {}, RUNS_DB, STUDY_YAML))

**barrier-kinematics**


In [ ]:
# barrier-kinematics
show_viz(_render_one('local:BarrierKinematics', {}, RUNS_DB, STUDY_YAML))

**energy-budget**


In [ ]:
# energy-budget
show_viz(_render_one('local:EnergyBudget', {}, RUNS_DB, STUDY_YAML))

**schematic-vesicle-3d**


In [ ]:
# schematic-vesicle-3d
show_viz(_render_one('local:SchematicVesicle3D', {}, RUNS_DB, STUDY_YAML))

**particles-3d**


In [ ]:
# particles-3d
show_viz(_render_one('local:Particles3D', {'study_slug': 'membrane-to-actin-displacement-feedback'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| mesh-quality-preserved-over-coupling-steps | kind=listener_path observable=membrane_min_z reduce=series | op monotonic_decreasing |


## Study: Rung 3 · Flexible Mem3DG boundary (`flexible-mem3dg-boundary`)

**Question.** With barrier_kind="flexible" (closed-icosphere Mem3DG mesh inside-pushed by bonded actin filaments), does the membrane deform under actin pressure and relax the contact-force burden compared to the rigid_movable rung?

**Purpose.** Spec §2.2 rung 3. The coupler runs full Mem3DG: actin tip positions raise contact_force on the inner surface; mem3dg integrates the resulting osmotic_offset; the vesicle inflates, distributing load across the mesh and shifting the F-V scatter point off the rigid-wall curve.

**Claim.** OBSERVED: under actin pressure the vesicle inflates. That load is redistributed across filaments / relaxed off the rigid curve is NOT yet quantified — inflation alone is not ratchet success.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `flexible-mem3dg-boundary` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 5 | — |
| `rung3-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `rung3-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `rung3-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `rung3-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `rung3-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `rung3-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `rung3-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `rung3-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `soft-coupling` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 5 | osmotic_force_scale=0.02 |
| `stiff-coupling` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 5 | osmotic_force_scale=0.1 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary`** — `spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary` (static spec file, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary = load_spec(REPO / 'pbg_membrane_actin_composite/composites/rung3_flexible_mem3dg_boundary.composite.yaml')
describe_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary)

In [ ]:
# === Edit parameters for composite 'rung3-flexible-mem3dg-boundary' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'actin_sim'  (local:ReaDDyProcess)
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['interval'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['box_size'] = [8.0, 8.0, 8.0]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['periodic'] = [False, False, False]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['species']['G'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['reactions'] = []
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['potentials'][0]['type'] = 'harmonic_repulsion'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['potentials'][0]['species1'] = 'G'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['potentials'][0]['species2'] = 'G'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['potentials'][0]['force_constant'] = 10.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['potentials'][0]['interaction_distance'] = 0.4
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][0] = [-0.44999999999999996, -0.44999999999999996, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][1] = [-0.15, -0.44999999999999996, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][2] = [0.15, -0.44999999999999996, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][3] = [0.44999999999999996, -0.44999999999999996, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][4] = [-0.44999999999999996, -0.15, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][5] = [-0.15, -0.15, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][6] = [0.15, -0.15, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][7] = [0.44999999999999996, -0.15, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_species']['F'] = 0.05
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_types'] = ['filament']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_bonds'][0]['type1'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_bonds'][0]['type2'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_bonds'][0]['force_constant'] = 200.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_bonds'][0]['length'] = 0.4
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_angles'][0]['type1'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_angles'][0]['type2'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_angles'][0]['type3'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_angles'][0]['force_constant'] = 30.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_angles'][0]['equilibrium_angle'] = 3.14159
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][0]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][0]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][0]['positions'][0] = [0.11055415967851333, 0.16666666666666669, 0.0]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][0]['positions'][1] = [0.33166247903554, 0.5000000000000001, 0.0]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][0]['positions'][2] = [0.5527707983925666, 0.8333333333333334, 0.0]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][0]['positions'][3] = [0.7738791177495933, 1.1666666666666667, 0.0]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][1]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][1]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][1]['positions'][0] = [-0.12771603607517107, 0.1, 0.11699835096806109]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][1]['positions'][1] = [-0.3831481082255132, 0.30000000000000004, 0.3509950529041833]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][1]['positions'][2] = [-0.6385801803758553, 0.5, 0.5849917548403054]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][1]['positions'][3] = [-0.8940122525261975, 0.7000000000000001, 0.8189884567764276]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][2]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][2]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][2]['positions'][0] = [0.01724058541736129, 0.03333333333333335, -0.19644757851232364]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][2]['positions'][1] = [0.051721756252083875, 0.10000000000000006, -0.589342735536971]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][2]['positions'][2] = [0.08620292708680645, 0.16666666666666674, -0.9822378925616182]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][2]['positions'][3] = [0.12068409792152904, 0.23333333333333345, -1.3751330495862657]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][3]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][3]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][3]['positions'][0] = [0.11998576148964023, -0.033333333333333305, 0.15650017868564897]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][3]['positions'][1] = [0.3599572844689207, -0.09999999999999992, 0.46950053605694697]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][3]['positions'][2] = [0.5999288074482011, -0.16666666666666652, 0.7825008934282448]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][3]['positions'][3] = [0.8399003304274816, -0.23333333333333314, 1.0955012507995427]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][4]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][4]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][4]['positions'][0] = [-0.1705573787464552, -0.1, -0.030169198781840885]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][4]['positions'][1] = [-0.5116721362393657, -0.30000000000000004, -0.09050759634552266]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][4]['positions'][2] = [-0.852786893732276, -0.5, -0.15084599390920442]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][4]['positions'][3] = [-1.1939016512251865, -0.7000000000000001, -0.2111843914728862]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][5]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][5]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][5]['positions'][0] = [0.09328065759228087, -0.16666666666666666, -0.059337518833987996]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][5]['positions'][1] = [0.27984197277684264, -0.5, -0.178012556501964]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][5]['positions'][2] = [0.4664032879614043, -0.8333333333333333, -0.29668759416993995]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][5]['positions'][3] = [0.652964603145966, -1.1666666666666667, -0.415362631837916]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['timestep'] = 0.005
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['observe_stride'] = 10

# process 'coupler'  (local:BrownianRatchetCoupler)
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['interval'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['closed_loop'] = True
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['coupling_mode'] = 'spherical'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['barrier_kind'] = 'flexible'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['barrier_initial_z'] = 2.5
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['barrier_drag'] = 5.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['contact_threshold'] = 0.3
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['force_constant'] = 5.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['osmotic_force_scale'] = 0.05

# process 'membrane_sim'  (local:Mem3DGProcess)
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['interval'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['mesh_type'] = 'icosphere'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['radius'] = 2.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['subdivision'] = 2
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['characteristic_timestep'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['tolerance'] = 1e-09
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['osmotic_model'] = 'constant'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['osmotic_pressure'] = 0.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['tension_modulus'] = 1.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['preferred_area_scale'] = 1.0

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: flexible-mem3dg-boundary ===
STUDY = 'flexible-mem3dg-boundary'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_flexible_mem3dg_boundary = 5
INTERVAL_flexible_mem3dg_boundary = 1.0
STEPS_rung3_flexible = 16
INTERVAL_rung3_flexible = 1.0
STEPS_rung3_flexible = 16
INTERVAL_rung3_flexible = 1.0
STEPS_rung3_flexible = 16
INTERVAL_rung3_flexible = 1.0
STEPS_rung3_flexible = 16
INTERVAL_rung3_flexible = 1.0
STEPS_rung3_flexible = 16
INTERVAL_rung3_flexible = 1.0
STEPS_rung3_flexible = 16
INTERVAL_rung3_flexible = 1.0
STEPS_rung3_flexible = 16
INTERVAL_rung3_flexible = 1.0
STEPS_rung3_flexible = 16
INTERVAL_rung3_flexible = 1.0
STEPS_soft_coupling = 5
INTERVAL_soft_coupling = 1.0
STEPS_stiff_coupling = 5
INTERVAL_stiff_coupling = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_flexible_mem3dg_boundary}, core=core)
            _comp.run(STEPS_flexible_mem3dg_boundary)
        print('ran', 'flexible-mem3dg-boundary', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_flexible_mem3dg_boundary, 'steps')
    except Exception as _e:
        print(f"flexible-mem3dg-boundary not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_rung3_flexible}, core=core)
            _comp.run(STEPS_rung3_flexible)
        print('ran', 'rung3-flexible', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_rung3_flexible, 'steps')
    except Exception as _e:
        print(f"rung3-flexible not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_rung3_flexible}, core=core)
            _comp.run(STEPS_rung3_flexible)
        print('ran', 'rung3-flexible', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_rung3_flexible, 'steps')
    except Exception as _e:
        print(f"rung3-flexible not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_rung3_flexible}, core=core)
            _comp.run(STEPS_rung3_flexible)
        print('ran', 'rung3-flexible', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_rung3_flexible, 'steps')
    except Exception as _e:
        print(f"rung3-flexible not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_rung3_flexible}, core=core)
            _comp.run(STEPS_rung3_flexible)
        print('ran', 'rung3-flexible', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_rung3_flexible, 'steps')
    except Exception as _e:
        print(f"rung3-flexible not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_rung3_flexible}, core=core)
            _comp.run(STEPS_rung3_flexible)
        print('ran', 'rung3-flexible', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_rung3_flexible, 'steps')
    except Exception as _e:
        print(f"rung3-flexible not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_rung3_flexible}, core=core)
            _comp.run(STEPS_rung3_flexible)
        print('ran', 'rung3-flexible', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_rung3_flexible, 'steps')
    except Exception as _e:
        print(f"rung3-flexible not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_rung3_flexible}, core=core)
            _comp.run(STEPS_rung3_flexible)
        print('ran', 'rung3-flexible', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_rung3_flexible, 'steps')
    except Exception as _e:
        print(f"rung3-flexible not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_rung3_flexible}, core=core)
            _comp.run(STEPS_rung3_flexible)
        print('ran', 'rung3-flexible', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_rung3_flexible, 'steps')
    except Exception as _e:
        print(f"rung3-flexible not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_soft_coupling}, core=core)
            _comp.run(STEPS_soft_coupling)
        print('ran', 'soft-coupling', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_soft_coupling, 'steps')
    except Exception as _e:
        print(f"soft-coupling not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_stiff_coupling}, core=core)
            _comp.run(STEPS_stiff_coupling)
        print('ran', 'stiff-coupling', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_stiff_coupling, 'steps')
    except Exception as _e:
        print(f"stiff-coupling not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**coupling-trace**


In [ ]:
# coupling-trace
show_viz(_render_one('local:CouplingTrace', {}, RUNS_DB, STUDY_YAML))

**backpressure-trace**


In [ ]:
# backpressure-trace
show_viz(_render_one('local:BackpressureTrace', {}, RUNS_DB, STUDY_YAML))

**population-trace**


In [ ]:
# population-trace
show_viz(_render_one('local:PopulationTrace', {}, RUNS_DB, STUDY_YAML))

**barrier-kinematics**


In [ ]:
# barrier-kinematics
show_viz(_render_one('local:BarrierKinematics', {}, RUNS_DB, STUDY_YAML))

**energy-budget**


In [ ]:
# energy-budget
show_viz(_render_one('local:EnergyBudget', {}, RUNS_DB, STUDY_YAML))

**membrane-volume-strain**


In [ ]:
# membrane-volume-strain
show_viz(_render_one('local:MembraneVolumeStrain', {}, RUNS_DB, STUDY_YAML))

**ratchet-event-rate**


In [ ]:
# ratchet-event-rate
show_viz(_render_one('local:RatchetEventRate', {}, RUNS_DB, STUDY_YAML))

**force-velocity-scatter**


In [ ]:
# force-velocity-scatter
show_viz(_render_one('local:ForceVelocityScatter', {}, RUNS_DB, STUDY_YAML))

**schematic-vesicle-3d**


In [ ]:
# schematic-vesicle-3d
show_viz(_render_one('local:SchematicVesicle3D', {}, RUNS_DB, STUDY_YAML))

**particles-3d**


In [ ]:
# particles-3d
show_viz(_render_one('local:Particles3D', {'study_slug': 'flexible-mem3dg-boundary'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| membrane-curvature-relaxes-load | kind=listener_path observable=membrane_volume reduce=first_and_last | op ratio_at_least ratio 1.1 |


## Study: Benchmark · Coupled force–velocity reproduction (`coupled-ratchet-cycle-fv-reproduction`)

**Question.** Does the closed-loop coupled cycle reproduce the Inoue-2015 concave→convex F-V transition as polymerization rate varies — the headline numerical benchmark for a flexible-membrane ratchet (spec §1.2)?

**Purpose.** Sweep growth_rate across the regime where the F-V curve shape flips: at low rates the curve is concave (monomer-delivery limited); at high rates it goes convex (filament-dominated). The flexible barrier is the only rung where this transition can emerge — rigid rungs give a single F-V scatter point per growth_rate, not the transition.

**Claim.** TARGET (not yet tested): the closed loop reproduces the Inoue-2015 concave→convex F-V transition. No benchmark comparison has been run.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `coupled-cycle-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `coupled-cycle-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `coupled-cycle-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `coupled-cycle-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `coupled-cycle-flexible` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `coupled-ratchet-cycle-fv-reproduction` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 5 | — |
| `high-growth` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `high-growth` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `high-growth` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `high-growth` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `high-growth` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 5 | growth_rate=8.0 |
| `low-growth` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `low-growth` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `low-growth` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `low-growth` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 16 | — |
| `low-growth` | `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary` | 5 | growth_rate=1.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary`** — `spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary` (static spec file, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary = load_spec(REPO / 'pbg_membrane_actin_composite/composites/rung3_flexible_mem3dg_boundary.composite.yaml')
describe_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary)

In [ ]:
# === Edit parameters for composite 'rung3-flexible-mem3dg-boundary' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'actin_sim'  (local:ReaDDyProcess)
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['interval'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['box_size'] = [8.0, 8.0, 8.0]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['periodic'] = [False, False, False]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['species']['G'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['reactions'] = []
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['potentials'][0]['type'] = 'harmonic_repulsion'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['potentials'][0]['species1'] = 'G'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['potentials'][0]['species2'] = 'G'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['potentials'][0]['force_constant'] = 10.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['potentials'][0]['interaction_distance'] = 0.4
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][0] = [-0.44999999999999996, -0.44999999999999996, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][1] = [-0.15, -0.44999999999999996, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][2] = [0.15, -0.44999999999999996, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][3] = [0.44999999999999996, -0.44999999999999996, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][4] = [-0.44999999999999996, -0.15, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][5] = [-0.15, -0.15, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][6] = [0.15, -0.15, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_particles']['G'][7] = [0.44999999999999996, -0.15, -0.15]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_species']['F'] = 0.05
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_types'] = ['filament']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_bonds'][0]['type1'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_bonds'][0]['type2'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_bonds'][0]['force_constant'] = 200.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_bonds'][0]['length'] = 0.4
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_angles'][0]['type1'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_angles'][0]['type2'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_angles'][0]['type3'] = 'F'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_angles'][0]['force_constant'] = 30.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['topology_angles'][0]['equilibrium_angle'] = 3.14159
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][0]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][0]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][0]['positions'][0] = [0.11055415967851333, 0.16666666666666669, 0.0]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][0]['positions'][1] = [0.33166247903554, 0.5000000000000001, 0.0]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][0]['positions'][2] = [0.5527707983925666, 0.8333333333333334, 0.0]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][0]['positions'][3] = [0.7738791177495933, 1.1666666666666667, 0.0]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][1]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][1]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][1]['positions'][0] = [-0.12771603607517107, 0.1, 0.11699835096806109]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][1]['positions'][1] = [-0.3831481082255132, 0.30000000000000004, 0.3509950529041833]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][1]['positions'][2] = [-0.6385801803758553, 0.5, 0.5849917548403054]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][1]['positions'][3] = [-0.8940122525261975, 0.7000000000000001, 0.8189884567764276]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][2]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][2]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][2]['positions'][0] = [0.01724058541736129, 0.03333333333333335, -0.19644757851232364]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][2]['positions'][1] = [0.051721756252083875, 0.10000000000000006, -0.589342735536971]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][2]['positions'][2] = [0.08620292708680645, 0.16666666666666674, -0.9822378925616182]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][2]['positions'][3] = [0.12068409792152904, 0.23333333333333345, -1.3751330495862657]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][3]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][3]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][3]['positions'][0] = [0.11998576148964023, -0.033333333333333305, 0.15650017868564897]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][3]['positions'][1] = [0.3599572844689207, -0.09999999999999992, 0.46950053605694697]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][3]['positions'][2] = [0.5999288074482011, -0.16666666666666652, 0.7825008934282448]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][3]['positions'][3] = [0.8399003304274816, -0.23333333333333314, 1.0955012507995427]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][4]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][4]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][4]['positions'][0] = [-0.1705573787464552, -0.1, -0.030169198781840885]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][4]['positions'][1] = [-0.5116721362393657, -0.30000000000000004, -0.09050759634552266]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][4]['positions'][2] = [-0.852786893732276, -0.5, -0.15084599390920442]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][4]['positions'][3] = [-1.1939016512251865, -0.7000000000000001, -0.2111843914728862]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][5]['type'] = 'filament'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][5]['particle_types'] = ['F', 'F', 'F', 'F']
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][5]['positions'][0] = [0.09328065759228087, -0.16666666666666666, -0.059337518833987996]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][5]['positions'][1] = [0.27984197277684264, -0.5, -0.178012556501964]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][5]['positions'][2] = [0.4664032879614043, -0.8333333333333333, -0.29668759416993995]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['initial_topologies'][5]['positions'][3] = [0.652964603145966, -1.1666666666666667, -0.415362631837916]
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['timestep'] = 0.005
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['actin_sim']['config']['observe_stride'] = 10

# process 'coupler'  (local:BrownianRatchetCoupler)
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['interval'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['closed_loop'] = True
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['coupling_mode'] = 'spherical'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['barrier_kind'] = 'flexible'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['barrier_initial_z'] = 2.5
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['barrier_drag'] = 5.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['contact_threshold'] = 0.3
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['force_constant'] = 5.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['coupler']['config']['osmotic_force_scale'] = 0.05

# process 'membrane_sim'  (local:Mem3DGProcess)
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['interval'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['mesh_type'] = 'icosphere'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['radius'] = 2.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['subdivision'] = 2
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['characteristic_timestep'] = 0.5
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['tolerance'] = 1e-09
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['osmotic_model'] = 'constant'
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['osmotic_pressure'] = 0.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['tension_modulus'] = 1.0
spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary['state']['membrane_sim']['config']['preferred_area_scale'] = 1.0

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: coupled-ratchet-cycle-fv-reproduction ===
STUDY = 'coupled-ratchet-cycle-fv-reproduction'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_coupled_cycle_flexible = 16
INTERVAL_coupled_cycle_flexible = 1.0
STEPS_coupled_cycle_flexible = 16
INTERVAL_coupled_cycle_flexible = 1.0
STEPS_coupled_cycle_flexible = 16
INTERVAL_coupled_cycle_flexible = 1.0
STEPS_coupled_cycle_flexible = 16
INTERVAL_coupled_cycle_flexible = 1.0
STEPS_coupled_cycle_flexible = 16
INTERVAL_coupled_cycle_flexible = 1.0
STEPS_coupled_ratchet_cycle_fv_reproduction = 5
INTERVAL_coupled_ratchet_cycle_fv_reproduction = 1.0
STEPS_high_growth = 16
INTERVAL_high_growth = 1.0
STEPS_high_growth = 16
INTERVAL_high_growth = 1.0
STEPS_high_growth = 16
INTERVAL_high_growth = 1.0
STEPS_high_growth = 16
INTERVAL_high_growth = 1.0
STEPS_high_growth = 5
INTERVAL_high_growth = 1.0
STEPS_low_growth = 16
INTERVAL_low_growth = 1.0
STEPS_low_growth = 16
INTERVAL_low_growth = 1.0
STEPS_low_growth = 16
INTERVAL_low_growth = 1.0
STEPS_low_growth = 16
INTERVAL_low_growth = 1.0
STEPS_low_growth = 5
INTERVAL_low_growth = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_coupled_cycle_flexible}, core=core)
            _comp.run(STEPS_coupled_cycle_flexible)
        print('ran', 'coupled-cycle-flexible', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_coupled_cycle_flexible, 'steps')
    except Exception as _e:
        print(f"coupled-cycle-flexible not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_coupled_cycle_flexible}, core=core)
            _comp.run(STEPS_coupled_cycle_flexible)
        print('ran', 'coupled-cycle-flexible', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_coupled_cycle_flexible, 'steps')
    except Exception as _e:
        print(f"coupled-cycle-flexible not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_coupled_cycle_flexible}, core=core)
            _comp.run(STEPS_coupled_cycle_flexible)
        print('ran', 'coupled-cycle-flexible', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_coupled_cycle_flexible, 'steps')
    except Exception as _e:
        print(f"coupled-cycle-flexible not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_coupled_cycle_flexible}, core=core)
            _comp.run(STEPS_coupled_cycle_flexible)
        print('ran', 'coupled-cycle-flexible', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_coupled_cycle_flexible, 'steps')
    except Exception as _e:
        print(f"coupled-cycle-flexible not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_coupled_cycle_flexible}, core=core)
            _comp.run(STEPS_coupled_cycle_flexible)
        print('ran', 'coupled-cycle-flexible', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_coupled_cycle_flexible, 'steps')
    except Exception as _e:
        print(f"coupled-cycle-flexible not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_coupled_ratchet_cycle_fv_reproduction}, core=core)
            _comp.run(STEPS_coupled_ratchet_cycle_fv_reproduction)
        print('ran', 'coupled-ratchet-cycle-fv-reproduction', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_coupled_ratchet_cycle_fv_reproduction, 'steps')
    except Exception as _e:
        print(f"coupled-ratchet-cycle-fv-reproduction not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_high_growth}, core=core)
            _comp.run(STEPS_high_growth)
        print('ran', 'high-growth', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_high_growth, 'steps')
    except Exception as _e:
        print(f"high-growth not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_high_growth}, core=core)
            _comp.run(STEPS_high_growth)
        print('ran', 'high-growth', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_high_growth, 'steps')
    except Exception as _e:
        print(f"high-growth not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_high_growth}, core=core)
            _comp.run(STEPS_high_growth)
        print('ran', 'high-growth', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_high_growth, 'steps')
    except Exception as _e:
        print(f"high-growth not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_high_growth}, core=core)
            _comp.run(STEPS_high_growth)
        print('ran', 'high-growth', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_high_growth, 'steps')
    except Exception as _e:
        print(f"high-growth not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_high_growth}, core=core)
            _comp.run(STEPS_high_growth)
        print('ran', 'high-growth', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_high_growth, 'steps')
    except Exception as _e:
        print(f"high-growth not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_low_growth}, core=core)
            _comp.run(STEPS_low_growth)
        print('ran', 'low-growth', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_low_growth, 'steps')
    except Exception as _e:
        print(f"low-growth not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_low_growth}, core=core)
            _comp.run(STEPS_low_growth)
        print('ran', 'low-growth', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_low_growth, 'steps')
    except Exception as _e:
        print(f"low-growth not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_low_growth}, core=core)
            _comp.run(STEPS_low_growth)
        print('ran', 'low-growth', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_low_growth, 'steps')
    except Exception as _e:
        print(f"low-growth not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_low_growth}, core=core)
            _comp.run(STEPS_low_growth)
        print('ran', 'low-growth', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_low_growth, 'steps')
    except Exception as _e:
        print(f"low-growth not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_membrane_actin_composite_composites_rung3_flexible_mem3dg_boundary, {'interval': INTERVAL_low_growth}, core=core)
            _comp.run(STEPS_low_growth)
        print('ran', 'low-growth', f'(pbg_membrane_actin_composite.composites.rung3_flexible_mem3dg_boundary) for', STEPS_low_growth, 'steps')
    except Exception as _e:
        print(f"low-growth not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**force-velocity-benchmark**


In [ ]:
# force-velocity-benchmark
show_viz(_render_one('local:ForceVelocityBenchmark', {'study_slug': 'coupled-ratchet-cycle-fv-reproduction'}, RUNS_DB, STUDY_YAML))

**force-velocity-scatter**


In [ ]:
# force-velocity-scatter
show_viz(_render_one('local:ForceVelocityScatter', {}, RUNS_DB, STUDY_YAML))

**coupling-trace**


In [ ]:
# coupling-trace
show_viz(_render_one('local:CouplingTrace', {}, RUNS_DB, STUDY_YAML))

**barrier-kinematics**


In [ ]:
# barrier-kinematics
show_viz(_render_one('local:BarrierKinematics', {}, RUNS_DB, STUDY_YAML))

**energy-budget**


In [ ]:
# energy-budget
show_viz(_render_one('local:EnergyBudget', {}, RUNS_DB, STUDY_YAML))

**membrane-volume-strain**


In [ ]:
# membrane-volume-strain
show_viz(_render_one('local:MembraneVolumeStrain', {}, RUNS_DB, STUDY_YAML))

**population-trace**


In [ ]:
# population-trace
show_viz(_render_one('local:PopulationTrace', {}, RUNS_DB, STUDY_YAML))

**ratchet-event-rate**


In [ ]:
# ratchet-event-rate
show_viz(_render_one('local:RatchetEventRate', {}, RUNS_DB, STUDY_YAML))

**schematic-vesicle-3d**


In [ ]:
# schematic-vesicle-3d
show_viz(_render_one('local:SchematicVesicle3D', {}, RUNS_DB, STUDY_YAML))

**particles-3d**


In [ ]:
# particles-3d
show_viz(_render_one('local:Particles3D', {'study_slug': 'coupled-ratchet-cycle-fv-reproduction'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| concave-to-convex-fv-transition-emerges | kind=listener_path observable=barrier_velocity reduce=mean | op ratio_at_least ratio 2.0 compared_to {'run': 'variant', 'variant': 'low-growth'} |


## Open decisions
- Which numerical tolerances define "matches Peskin / Inoue"?
- Spherical mesh-vertex feedback or planar wall_radius for the canonical F-V run?
- Where do the Peskin (1993) and Inoue (2015) reference curves come from for the overlays?
